# Fitbit fitness tracker analysis: Data cleaning

## 1. Overview
This notebook establishes an end-to-end data cleaning and feature engineering pipeline for the Fitbit fitness tracker dataset. Drawing from raw multi-table inputs (`dailyActivity`, `sleepDay`, `hourlyIntensity`, `hourlySteps`, and `hourlyCalories`), this script handles timestamp normalization, duplicate purging, granularity alignment, and structural left joins to prepare a pristine, analysis-ready dataset. The clean dataframes are structured for downstream exploratory data analysis (EDA). 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Loading daily activity and sleep primary datasets
df_activity = pd.read_csv("dailyActivity_merged.csv")
df_sleep = pd.read_csv("sleepDay_merged.csv")

# Loading hourly activity, intensity, and calorie datasets
df_hourly_intensity = pd.read_csv("hourlyIntensities_merged.csv")
df_hourly_calories = pd.read_csv("hourlyCalories_merged.csv")
df_hourly_steps = pd.read_csv("hourlySteps_merged.csv")

## 2. Initial data inspection
Examining individual datasets for the following to flag anomalies:
* dataset shapes
* feature data types
* missing values
* duplicate records

In [ ]:
# Inspecting shapes and data types for all tables
for name, df in [('Activity', df_activity), ('Sleep', df_sleep), 
                   ('Intensity', df_hourly_intensity), ('Calories', df_hourly_calories), ('Steps', df_hourly_steps)]:
    print(f"--- {name} ---")
    print(f"Shape: {df.shape}")
    print(df.dtypes, "\n")

# Auditing missing values across inputs
for name, df in [('Activity', df_activity), ('Sleep', df_sleep), 
                   ('Intensity', df_hourly_intensity), ('Calories', df_hourly_calories), ('Steps', df_hourly_steps)]:
    print(f"--- {name} ---")
    print(f"Missing Values:\n{df.isna().sum()}\n")

# Auditing duplicate rows across inputs
for name, df in [('Activity', df_activity), ('Sleep', df_sleep), 
                   ('Intensity', df_hourly_intensity), ('Calories', df_hourly_calories), ('Steps', df_hourly_steps)]:
    print(f"--- {name} ---")
    print(f"Duplicate Rows:\n{df.duplicated().sum()}\n")

# Inspecting exact duplicate rows in sleep data based on 'Id' and 'SleepDay' keys
duplicates = df_sleep[df_sleep.duplicated(subset=["Id", "SleepDay"], keep=False)]
print("Sample Sleep Duplicates:\n", duplicates.sort_values(["Id", "SleepDay"]))

--- Activity ---
Shape: (940, 15)
Id                            int64
ActivityDate                    str
TotalSteps                    int64
TotalDistance               float64
TrackerDistance             float64
LoggedActivitiesDistance    float64
VeryActiveDistance          float64
ModeratelyActiveDistance    float64
LightActiveDistance         float64
SedentaryActiveDistance     float64
VeryActiveMinutes             int64
FairlyActiveMinutes           int64
LightlyActiveMinutes          int64
SedentaryMinutes              int64
Calories                      int64
dtype: object 

--- Sleep ---
Shape: (413, 5)
Id                    int64
SleepDay                str
TotalSleepRecords     int64
TotalMinutesAsleep    int64
TotalTimeInBed        int64
dtype: object 

--- Intensity ---
Shape: (22099, 4)
Id                    int64
ActivityHour            str
TotalIntensity        int64
AverageIntensity    float64
dtype: object 

--- Calories ---
Shape: (22099, 3)
Id              int64
Act

## 3. "Daily data" - Cleaning & Preprocessing
* **Timestamp normalization:** Converting string dates into standardized `datetime` format.
* **Handling duplicates:** Identifying and dropping exact-row duplicates within the sleep tracking dataset.

In [ ]:
# Standardizing date columns to datetime objects
df_activity["ActivityDate"] = pd.to_datetime(df_activity["ActivityDate"], format="%m/%d/%Y")
df_sleep["SleepDay"] = pd.to_datetime(df_sleep["SleepDay"], format="%m/%d/%Y")

# Removing exact-row duplicate records from sleep dataset
# Reasoning - duplicates are highly unlikely to be different sleep session due to identical durations
df_sleep.drop_duplicates(subset=["Id", "SleepDay"], keep='first', inplace=True)
print("Duplicates in Sleep after cleaning:", df_sleep.duplicated().sum())

# Verifying if all sleep IDs are a strict subset of the activity dataset
all_sleep_in_activity = df_sleep['Id'].isin(df_activity['Id']).all()
print("All sleep IDs are present in activity data:", all_sleep_in_activity)

# Verifying unique user counts
print("Unique IDs in Activity:", df_activity['Id'].nunique())
print("Unique IDs in Sleep:", df_sleep['Id'].nunique())

Duplicates in Sleep after cleaning: 0
All sleep IDs are present in activity data: True
Unique IDs in Activity: 33
Unique IDs in Sleep: 24


## 4. "Daily data" - Feature engineering
* extracting date/time attributes (`Day_of_the_week`, `Weekend`)
* deriving clinical/behavioral sleep features (`HoursAsleep`, `HoursInBed`, and `SleepEfficiency`).

In [11]:
# Extracting temporal features
df_activity['Day_of_the_week'] = df_activity['ActivityDate'].dt.day_name()
df_activity['Weekend'] = df_activity['Day_of_the_week'].isin(['Saturday', 'Sunday'])

# Engineering sleep metrics
df_sleep['HoursAsleep'] = df_sleep['TotalMinutesAsleep'] / 60
df_sleep['HoursInBed'] = df_sleep['TotalTimeInBed'] / 60
df_sleep['SleepEfficiency'] = df_sleep['HoursAsleep'] / df_sleep['HoursInBed']

## 5. "Daily" and "Hourly datasets" - temporal overlap audit
Checking for timeframe overlap for succesful merging.

In [ ]:
# Checking date range overlaps for daily/sleep data 
print("Activity Date Range:", df_activity["ActivityDate"].min(), "to", df_activity["ActivityDate"].max())
print("Sleep Date Range:", df_sleep["SleepDay"].min(), "to", df_sleep["SleepDay"].max())

# Overlap outcome
activity_dates = set(df_activity['ActivityDate'].dt.date)
sleep_dates = set(df_sleep['SleepDay'].dt.date)

if activity_dates == sleep_dates:
    print("Daily and sleep date sets match perfectly.")
elif activity_dates.isdisjoint(sleep_dates):
    print("Warning: No temporal overlap between activity and sleep datasets.")
else:
    print("Partial overlap: Sleep dates intersect with activity dates.")

# Checking timestamp equality for hourly data
set_intensity = set(df_hourly_intensity['ActivityHour'])
set_calories = set(df_hourly_calories['ActivityHour'])
set_steps = set(df_hourly_steps['ActivityHour'])

# Overlap outcome
if set_intensity == set_calories == set_steps:
    print("Hourly timestamp sets match perfectly.")
else:
    print("Warning: Discrepancy found in hourly timestamp sets.")

Activity Date Range: 2016-04-12 00:00:00 to 2016-05-12 00:00:00
Sleep Date Range: 2016-04-12 00:00:00 to 2016-05-12 00:00:00
Daily and sleep date sets match perfectly.
Hourly timestamp sets match perfectly.


## 6. "Daily dataset" - Merging
Executing a left join on user `Id` and date (`ActivityDate` matching `SleepDay`) to preserve all 33 user activity profiles while maintaining structural alignment with the 24-user sleep subset.

In [ ]:
# Merging activity and sleep tables via left join
df_merged_daily = pd.merge(
    df_activity, df_sleep, 
    left_on=['Id', 'ActivityDate'], 
    right_on=['Id', 'SleepDay'], 
    how='left'
)

# Cleaning up redundant key columns and sort chronologically
df_merged_daily = df_merged_daily.drop(columns=['SleepDay'])
df_merged_daily = df_merged_daily.sort_values(['Id', 'ActivityDate']).reset_index(drop=True)

## 7. "Hourly datasets" - Preprocessing & Consolidation
* **Timestamp normalization:** Converting string dates into standardized `datetime` format.
* **Feature engineering:** Extracting date/hour attributes (`Date`, `Hour`, `DayOfWeek`, `IsWeekend`).
* **Merging hourly datasets:** Left joining on user `Id` and `ActivityHour`.

In [12]:
# Parse full timestamps and extract hourly components
for df in [df_hourly_intensity, df_hourly_steps, df_hourly_calories]:
    df['ActivityHour'] = pd.to_datetime(df['ActivityHour'], format='%m/%d/%Y %I:%M:%S %p')
    df['Date'] = df['ActivityHour'].dt.date
    df['Hour'] = df['ActivityHour'].dt.hour
    df['DayOfWeek'] = df['ActivityHour'].dt.day_name()
    df['IsWeekend'] = df['ActivityHour'].dt.dayofweek >= 5

# Verify if all hourly files share identical ID sets
identical_hourly_ids = (set(df_hourly_intensity['Id']) == set(df_hourly_calories['Id']) == set(df_hourly_steps['Id']))
print("Identical IDs across all hourly datasets:", identical_hourly_ids)

# Sequentially merge hourly datasets on user ID and timestamp
df_merged_hourly = df_hourly_intensity.merge(df_hourly_steps, on=['Id', 'ActivityHour'], how='left')
df_merged_hourly = df_merged_hourly.merge(df_hourly_calories, on=['Id', 'ActivityHour'], how='left')

# Drop trailing redundant columns from intermediate merges
cols_to_drop = [c for c in df_merged_hourly.columns if c.endswith('_x') or c.endswith('_y')]
df_merged_hourly = df_merged_hourly.drop(columns=cols_to_drop)
df_merged_hourly = df_merged_hourly.sort_values(['Id', 'ActivityHour']).reset_index(drop=True)

Identical IDs across all hourly datasets: True


## 7. Final checks & export
* Performing a final validation pass on shapes and missing values.
* Exporting the cleaned datasets to CSV for future EDA.

In [13]:
# Final validation checks
print("Daily Clean Shape:", df_merged_daily.shape)
print("Hourly Clean Shape:", df_merged_hourly.shape)
print("\nDaily Missing Values:\n", df_merged_daily.isna().sum())
print("\nHourly Missing Values:\n", df_merged_hourly.isna().sum())

# Exporting clean files for the EDA workflow
df_merged_daily.to_csv("daily_merged_clean.csv", index=False)
df_merged_hourly.to_csv("hourly_merged_clean.csv", index=False)
print("\nPipeline execution complete: Cleaned datasets exported successfully.")

Daily Clean Shape: (940, 23)
Hourly Clean Shape: (22099, 10)

Daily Missing Values:
 Id                            0
ActivityDate                  0
TotalSteps                    0
TotalDistance                 0
TrackerDistance               0
LoggedActivitiesDistance      0
VeryActiveDistance            0
ModeratelyActiveDistance      0
LightActiveDistance           0
SedentaryActiveDistance       0
VeryActiveMinutes             0
FairlyActiveMinutes           0
LightlyActiveMinutes          0
SedentaryMinutes              0
Calories                      0
Day_of_the_week               0
Weekend                       0
TotalSleepRecords           530
TotalMinutesAsleep          530
TotalTimeInBed              530
HoursAsleep                 530
HoursInBed                  530
SleepEfficiency             530
dtype: int64

Hourly Missing Values:
 Id                  0
ActivityHour        0
TotalIntensity      0
AverageIntensity    0
StepTotal           0
Calories            0
Date     